# Extended Data Fig. 3j — *BMPR1A* after BMPR1A knockdown

Author: Tianlei He. Ported from `Compare BMPR1A level bulk seq(only BMPR1Akd and SCRAMBLE).ipynb`.

The panel plots *BMPR1A* CPM in BMPR1A-knockdown and scrambled-guide control cells, three replicates
each, as bars (mean ± s.d.) with replicates as points. Significance is a two-sided Welch's t-test; the
Holm adjustment is applied over this single comparison and so leaves p unchanged. CPM values are used
exactly as computed by the sequencing provider; nothing here re-normalizes them.

**Changes from the original notebook**
1. Input is the GEO series GSE347564 CPM table instead of the provider's per-order matrix, which the
   original extracted from a zip archive. Column names change accordingly: `F3FV83_1–3_cpm` →
   `BMPR1Akd_rep1–3`, `F3FV83_4–6_cpm` → `SCRAMBLE_rep1–3`.
2. Paths are relative to the repository, with a check that the input table exists; outputs go to `bulkseq/output/`.
3. Removed cells that the panel does not use: the zip extraction, and the pooled-CPM histogram with the
   mean-CPM ≥ 2 filtered-table export (the plot reads the unfiltered row). The line that printed
   whether *BMPR1A* passed that filter is removed with it.
4. Added the last cell, which writes the plotted values to `bulkseq/output/ed3j_plotted_values.tsv`.

In [ ]:
# %pip install pandas matplotlib numpy seaborn scipy statsmodels

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bulkseq" / "README.md").is_file())
CPM_TABLE = Path(os.environ.get(
    "BULK_CPM_TABLE", REPO / "data" / "GSE347564" / "GSE347564_bulk_RNAseq_cpm_all_samples.tsv.gz"))
OUT = REPO / "bulkseq" / "output"
OUT.mkdir(parents=True, exist_ok=True)

out_barplot_png = OUT / "BMPR1A_BMPR1Akd_vs_SCRAMBLE.png"
out_barplot_pdf = OUT / "BMPR1A_BMPR1Akd_vs_SCRAMBLE.pdf"

annotation_columns = ["gene_id", "gene_name", "gene_biotype"]

GROUPS = {
    "BMPR1Akd": [
        "BMPR1Akd_rep1",
        "BMPR1Akd_rep2",
        "BMPR1Akd_rep3",
    ],
    "SCRAMBLE": [
        "SCRAMBLE_rep1",
        "SCRAMBLE_rep2",
        "SCRAMBLE_rep3",
    ],
}

cpm_cols = [c for cols in GROUPS.values() for c in cols]

TARGET_GENE = "BMPR1A"
EXPECTED_GENE_ID = None  # fill in if you want to enforce a specific Ensembl ID

In [ ]:
if not CPM_TABLE.is_file():
    raise FileNotFoundError(f"CPM table not found: {CPM_TABLE}. See bulkseq/README.md.")
plot_df = pd.read_csv(CPM_TABLE, sep="\t")
missing_cols = [c for c in annotation_columns + cpm_cols if c not in plot_df.columns]
if missing_cols:
    raise ValueError(f"Matrix missing required columns: {missing_cols}")

print(f"Loaded matrix: {CPM_TABLE}")
print(f"Rows: {len(plot_df)}")
plot_df.head()

In [ ]:
sample_map = [{"group": g, "cpm_column": c} for g, cols in GROUPS.items() for c in cols]
display(pd.DataFrame(sample_map))

hits = plot_df[plot_df["gene_name"].fillna("").str.upper() == TARGET_GENE.upper()]
if hits.empty:
    raise ValueError(f"No row with gene_name == {TARGET_GENE!r}.")
if len(hits) > 1:
    print(f"WARNING: {len(hits)} rows match {TARGET_GENE}; using the first.")
    display(hits[["gene_id", "gene_name", "gene_biotype"] + cpm_cols].head(10))

gene = hits.iloc[0]
print(
    f"Using: gene_id={gene['gene_id']!r}, gene_name={gene['gene_name']!r}, biotype={gene.get('gene_biotype', 'n/a')!r}"
)

if EXPECTED_GENE_ID and str(gene["gene_id"]).upper() != EXPECTED_GENE_ID.upper():
    print(
        f"WARNING: gene_id {gene['gene_id']!r} != EXPECTED_GENE_ID {EXPECTED_GENE_ID!r} - verify annotation."
    )

vals = np.array([float(gene[c]) for c in cpm_cols], dtype=float)
if not np.isfinite(vals).all():
    bad = [cpm_cols[i] for i, v in enumerate(vals) if not np.isfinite(v)]
    raise ValueError(f"Non-finite CPM in: {bad}")
if (vals < 0).any():
    raise ValueError("Negative CPM values found for target gene.")

In [ ]:
rows_out = []
rec = {"gene_name": gene["gene_name"], "gene_id": gene["gene_id"]}
for gname, cols in GROUPS.items():
    group_vals = np.array([float(gene[c]) for c in cols], dtype=float)
    rec[f"mean_cpm::{gname}"] = float(np.mean(group_vals))
    rec[f"std_cpm::{gname}"] = float(np.std(group_vals, ddof=1)) if len(group_vals) > 1 else 0.0
    rec[f"replicate_cpms::{gname}"] = list(group_vals)
rows_out.append(rec)

wide_tbl = pd.DataFrame(rows_out)
display(wide_tbl)

In [ ]:
def p_to_stars(p):
    if p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"

long_rows = []
for gname, cols in GROUPS.items():
    for c in cols:
        long_rows.append(
            {
                "gene_name": TARGET_GENE,
                "group": gname,
                "cpm": float(gene[c]),
            }
        )

long_df = pd.DataFrame(long_rows)

group_names = list(GROUPS.keys())
vals_1 = long_df.loc[long_df["group"] == group_names[0], "cpm"].to_numpy()
vals_2 = long_df.loc[long_df["group"] == group_names[1], "cpm"].to_numpy()

# two-sided Welch t-test + Holm adjustment
_, p_raw = ttest_ind(vals_1, vals_2, equal_var=False, alternative="two-sided")
_, p_holm, _, _ = multipletests([p_raw], method="holm")
p_adj = p_holm[0]
stars = p_to_stars(p_adj)

fig, ax = plt.subplots(figsize=(6, 5.5))
sns.barplot(
    data=long_df,
    x="gene_name",
    y="cpm",
    hue="group",
    order=[TARGET_GENE],
    errorbar="sd",
    capsize=0.08,
    ax=ax,
    palette=["#1b9e77", "#7570b3"],
)
sns.stripplot(
    data=long_df,
    x="gene_name",
    y="cpm",
    hue="group",
    order=[TARGET_GENE],
    dodge=True,
    jitter=0.15,
    ax=ax,
    palette=["black", "black"],
    size=8,
    alpha=1,
    linewidth=1,
    edgecolor="white",
    legend=False,
    zorder=10,
)

y_max = long_df["cpm"].max()
y_min = long_df["cpm"].min()
y_range = max(y_max - y_min, 1.0)

bracket_y = y_max + 0.10 * y_range
bracket_h = 0.05 * y_range
x1, x2 = -0.2, 0.2

ax.plot(
    [x1, x1, x2, x2],
    [bracket_y, bracket_y + bracket_h, bracket_y + bracket_h, bracket_y],
    lw=1.5,
    c="black",
)
ax.text(
    (x1 + x2) / 2,
    bracket_y + bracket_h,
    stars,
    ha="center",
    va="bottom",
    fontsize=16,
)

ax.set_xlabel("")
ax.set_ylabel("CPM")
ax.set_title("BMPR1A - BMPR1Akd vs SCRAMBLEkd")
ax.set_ylim(0, bracket_y + bracket_h + 0.15 * y_range)
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()
fig.savefig(out_barplot_png, dpi=300, bbox_inches="tight")
plt.savefig(out_barplot_pdf, format="pdf", dpi=300)
plt.show()

print(f"raw p-value = {p_raw:.4g}")
print(f"Holm-adjusted p-value = {p_adj:.4g}")
print(f"significance = {stars}")

print("Significance legend:")
print("ns    : p >= 0.05")
print("*     : p < 0.05")
print("**    : p < 0.01")
print("***   : p < 0.001")
print("****  : p < 0.0001")

print(f"Saved PNG: {out_barplot_png}")
print(f"Saved PDF: {out_barplot_pdf}")

In [ ]:
plotted_rows = [(gene["gene_id"], TARGET_GENE, g, c, float(gene[c])) for g, cols in GROUPS.items() for c in cols]

plotted = pd.DataFrame(
    [{"gene_id": row_gene_id, "gene_name": sym, "group": g, "sample": c, "cpm": v}
     for (row_gene_id, sym, g, c, v) in plotted_rows]
)
plotted.to_csv(OUT / "ed3j_plotted_values.tsv", sep="\t", index=False)
print(f"ED 3j: wrote {len(plotted)} plotted values to bulkseq/output/ed3j_plotted_values.tsv")